In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Admin\AppData\Local\Temp\ipykernel_17424\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Internship\Traditinal RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: YOLO_MultiClass_Pipeline_Solution.pdf
  ✓ Loaded 2 pages

Total documents loaded: 2


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T10:47:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T10:47:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\YOLO_MultiClass_Pipeline_Solution.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'file_type': 'pdf'}, page_content="YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe performance degradation due to\nredundant preprocessing, double inference forward passes, and disjointed Non-Maximum Suppression (NMS)\nexecution. The architectural solution is to merge t

In [14]:
### Text splitting get into chunsks

def split_documnets(documents,chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
        
        # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
        return split_docs


In [15]:
chunks=split_documnets(all_pdf_documents)
chunks

Split 2 documents into 7 chunks

Example chunk:
Content: YOLO Optimization Architecture Whitepaper
Page 1
 Solutions for YOLO Multi-Model Deployment
 Issues
 Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified
 Mu...
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T10:47:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T10:47:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\YOLO_MultiClass_Pipeline_Solution.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T10:47:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T10:47:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\YOLO_MultiClass_Pipeline_Solution.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'file_type': 'pdf'}, page_content='YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe performance degradation due to\nredundant preprocessing, double inference forward passes, and disjointed Non-Maximum Suppression (NMS)\nexecution. The architectural solution is to merge t

### Embedding and vectorDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1973.70it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_17424\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [18]:
# Instantiate
embedding_manager = EmbeddingManager()

# Check dimension directly:
dim = embedding_manager.model.get_sentence_embedding_dimension()
print("Direct check - Embedding Dimension:", dim)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5817.50it/s]


Model loaded successfully. Embedding dimension: 384
Direct check - Embedding Dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_17424\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
C:\Users\Admin\AppData\Local\Temp\ipykernel_17424\3098896795.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = embedding_manager.model.get_sentence_embedding_dimension()


### Vector store


In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 7


In [20]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T10:47:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T10:47:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\YOLO_MultiClass_Pipeline_Solution.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'file_type': 'pdf'}, page_content='YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe performance degradation due to\nredundant preprocessing, double inference forward passes, and disjointed Non-Maximum Suppression (NMS)\nexecution. The architectural solution is to merge t

In [21]:
texts=[doc.page_content for doc in chunks] 
texts

['YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe performance degradation due to\nredundant preprocessing, double inference forward passes, and disjointed Non-Maximum Suppression (NMS)\nexecution. The architectural solution is to merge the independent datasets and retrain a single multi-class model.\nWhen edge resource constraints require temporary pipeline adjustments, strict software wrapper layers must handle\nshared array preprocessing, asynchronous execution threads, or consolidated NMS tensors.\nIdentified Problem (Multi-.pt\nEngine)\nArchitectural Solution Blueprint\n1. Double Preprocessing\nEach network runs isolated resize\nand image normalization routines\nindependently, duplicating memory\nmodifications.',
 "Engine)\nArchit

In [22]:
texts=[doc.page_content for doc in chunks]
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 7 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]

Generated embeddings with shape: (7, 384)
Adding 7 documents to vector store...
Successfully added 7 documents to vector store
Total documents in collection: 14


In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [24]:
rag_retriever

In [25]:
rag_retriever.retrieve("What is the single-pass preprocessing solution for YOLO?")

Retrieving documents for query: 'What is the single-pass preprocessing solution for YOLO?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_194643b2_0',
  'content': 'YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe performance degradation due to\nredundant preprocessing, double inference forward passes, and disjointed Non-Maximum Suppression (NMS)\nexecution. The architectural solution is to merge the independent datasets and retrain a single multi-class model.\nWhen edge resource constraints require temporary pipeline adjustments, strict software wrapper layers must handle\nshared array preprocessing, asynchronous execution threads, or consolidated NMS tensors.\nIdentified Problem (Multi-.pt\nEngine)\nArchitectural Solution Blueprint\n1. Double Preprocessing\nEach network runs isolated resize\nand image normalization routines\nindependently, duplicating memo

In [26]:
rag_retriever.retrieve("What solution is suggested for handling fragmented Non-Maximum Suppression (NMS) across decoupled model spaces?")

Retrieving documents for query: 'What solution is suggested for handling fragmented Non-Maximum Suppression (NMS) across decoupled model spaces?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### integration vectorDB context pipeline with LLM output

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# Load key securely from .env

import os
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize ChatGroq with the active Llama model
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024,
)

def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve relevant context
    results = retriever.retrieve(query, top_k=top_k)
    
    # Handle dict vs Document objects dynamically
    context_chunks = []
    for doc in results:
        if isinstance(doc, dict):
            context_chunks.append(doc.get("content", ""))
        elif hasattr(doc, "page_content"):
            context_chunks.append(doc.page_content)
        else:
            context_chunks.append(str(doc))
            
    context = "\n\n".join([c for c in context_chunks if c])
    
    if not context.strip():
        return "No relevant context found to answer the question."

    # Direct f-string prompt (avoid redundant .format())
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content

# Run pipeline
answer = rag_simple(
    "What is the single-pass preprocessing solution for YOLO?",
    rag_retriever,
    llm
)
print(answer)

Retrieving documents for query: 'What is the single-pass preprocessing solution for YOLO?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The solution is to perform **one shared resize‑and‑normalize step** on the input image—creating a single pre‑processed tensor that is then fed to the (merged) YOLO network (or to both models). By doing the preprocessing only once, you eliminate duplicate memory operations and enable a unified, single‑pass inference pipeline.


In [33]:
from groq import Groq
import os

client = Groq(api_key=grooq_api_key)
models = client.models.list()
print("Available models on your account:")
for m in models.data:
    print(f"- {m.id}")

Available models on your account:
- qwen/qwen3.6-27b
- openai/gpt-oss-20b
- whisper-large-v3-turbo
- canopylabs/orpheus-v1-english
- groq/compound-mini
- allam-2-7b
- meta-llama/llama-prompt-guard-2-22m
- openai/gpt-oss-120b
- qwen/qwen3.8-27b
- whisper-large-v3
- canopylabs/orpheus-arabic-saudi
- openai/gpt-oss-safeguard-20b
- groq/compound
- meta-llama/llama-prompt-guard-2-86m


In [35]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the single-pass preprocessing solution for YOLO?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the single-pass preprocessing solution for YOLO?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: A single shared preprocessing pipeline that resizes and normalizes the input image **once**, then feeds that single pre‑processed tensor into the unified multi‑class YOLO model—eliminating duplicate resize/normalization steps.
Sources: [{'source': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'page': 0, 'score': 0.10676693916320801, 'preview': 'YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutive Summary: Running two separate .pt models simultaneously causes severe pe...'}, {'source': 'YOLO_MultiClass_Pipeline_Solution.pdf', 'page': 0, 'score': 0.10676693916320801, 'preview': 'YOLO Optimization Architecture Whitepaper\nPage 1\n Solutions for YOLO Multi-Model Deployment\n Issues\n Comprehensive Architectural Blueprint to Migrate from Two Model Pipelines into a Single Unified\n Multi-Class Network\nExecutiv